In [1]:
# initiate spark Boiler plate code
import pyspark
from pyspark.sql import SparkSession
spark  = SparkSession.builder.appName("joins").getOrCreate()

In [2]:
from pyspark.sql.functions import *

In [ ]:
bike_data = spark.read.csv("C:\\Users\\DELL\\Desktop\\AI_DATA_ENGINEERING\\Data_Sets\\Bike_Data.csv" , header = True , inferSchema = True , escape = '"')

In [4]:
bike_data.show(6, False)

+-------------+-------------+-----------------------------------------+----------------+--------------+-----------------------+-----+-------------------+---------+---------+--------+
|Region       |Country      |Customer                                 |Business Segment|Category      |Model                  |Color|SalesDate          |ListPrice|UnitPrice|OrderQty|
+-------------+-------------+-----------------------------------------+----------------+--------------+-----------------------+-----+-------------------+---------+---------+--------+
|North America|United States|Professional Containers and Packaging Co.|Clothing        |Tights        |Women's Tights         |Black|2018-08-22 00:00:00|74.99    |44.99    |1       |
|North America|United States|Professional Containers and Packaging Co.|Clothing        |Gloves        |Full-Finger Gloves     |Black|2018-08-17 00:00:00|37.99    |22.79    |2       |
|North America|United States|Professional Containers and Packaging Co.|Bikes         

In [7]:
bike_data.printSchema()

root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Customer: string (nullable = true)
 |-- Business Segment: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- SalesDate: timestamp (nullable = true)
 |-- ListPrice: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- OrderQty: integer (nullable = true)



In [8]:
b_df = bike_data.select("Region", "Country" ,"Business Segment", "ListPrice" ,"UnitPrice","OrderQty" )

In [9]:
b_df.show(5)

+-------------+-------------+----------------+---------+---------+--------+
|       Region|      Country|Business Segment|ListPrice|UnitPrice|OrderQty|
+-------------+-------------+----------------+---------+---------+--------+
|North America|United States|        Clothing|    74.99|    44.99|       1|
|North America|United States|        Clothing|    37.99|    22.79|       2|
|North America|United States|           Bikes|  2294.99|  1229.46|       1|
|North America|United States|        Clothing|    49.99|    28.84|       6|
|North America|United States|        Clothing|    37.99|    22.79|       4|
+-------------+-------------+----------------+---------+---------+--------+
only showing top 5 rows


In [10]:
b_df.count(),len(b_df.columns)

(60920, 6)

In [ ]:
# derived Columns 
#In Pandas
bike_data['Cost'] = np.round(bike_data['UnitPrice'] * bike_data['OrderQty'] , 1) # 1) sales = Listprice * Unitprice
bike_data['Sales'] = np.round(bike_data['ListPrice'] * bike_data['OrderQty'] , 2) # 2) Cost = unitprice * Orderqty
bike_data['Profit'] = np.round(bike_data['Sales'] * bike_data['Cost'], 3)  # 3) Profit = Sales - Cost¶

In [12]:
# in pyspark
b_df = b_df.withColumn("Cost",col("UnitPrice") * col("OrderQty"))
b_df = b_df.withColumn("Sales",col("ListPrice") * col("OrderQty"))
b_df = b_df.withColumn("Profit",col("Sales") * col("Cost"))

In [13]:
b_df.count(),len(b_df.columns)

(60920, 9)

In [15]:
b_df.show(2)

+-------------+-------------+----------------+---------+---------+--------+-----+-----+---------+
|       Region|      Country|Business Segment|ListPrice|UnitPrice|OrderQty| Cost|Sales|   Profit|
+-------------+-------------+----------------+---------+---------+--------+-----+-----+---------+
|North America|United States|        Clothing|    74.99|    44.99|       1|44.99|74.99|3373.8001|
|North America|United States|        Clothing|    37.99|    22.79|       2|45.58|75.98|3463.1684|
+-------------+-------------+----------------+---------+---------+--------+-----+-----+---------+
only showing top 2 rows


In [ ]:
# bronze / Silver/ gold sales < 5000 ---> bronze , 5-15K silver , morethan 15k gold

In [16]:
b_df = b_df.withColumn("Profit", col("Profit").cast("integer"))

In [17]:
b_df.printSchema()

root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Business Segment: string (nullable = true)
 |-- ListPrice: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Profit: integer (nullable = true)



In [18]:
b_df.show(5)

+-------------+-------------+----------------+---------+---------+--------+-------+-------+-------+
|       Region|      Country|Business Segment|ListPrice|UnitPrice|OrderQty|   Cost|  Sales| Profit|
+-------------+-------------+----------------+---------+---------+--------+-------+-------+-------+
|North America|United States|        Clothing|    74.99|    44.99|       1|  44.99|  74.99|   3373|
|North America|United States|        Clothing|    37.99|    22.79|       2|  45.58|  75.98|   3463|
|North America|United States|           Bikes|  2294.99|  1229.46|       1|1229.46|2294.99|2821598|
|North America|United States|        Clothing|    49.99|    28.84|       6| 173.04| 299.94|  51901|
|North America|United States|        Clothing|    37.99|    22.79|       4|  91.16| 151.96|  13852|
+-------------+-------------+----------------+---------+---------+--------+-------+-------+-------+
only showing top 5 rows


In [20]:
b_df = b_df.withColumn(
    "Sales_Grp",
    when(b_df["Sales"] < 5000, "Bronze")
    .when(b_df["Sales"] < 15000, "Silver")
    .when(b_df["Sales"] < 20000, "Gold")
    .otherwise("Platinum")
)

In [22]:
b_df.show(10)

+-------------+-------------+----------------+---------+---------+--------+------------------+------------------+--------+---------+
|       Region|      Country|Business Segment|ListPrice|UnitPrice|OrderQty|              Cost|             Sales|  Profit|Sales_Grp|
+-------------+-------------+----------------+---------+---------+--------+------------------+------------------+--------+---------+
|North America|United States|        Clothing|    74.99|    44.99|       1|             44.99|             74.99|    3373|   Bronze|
|North America|United States|        Clothing|    37.99|    22.79|       2|             45.58|             75.98|    3463|   Bronze|
|North America|United States|           Bikes|  2294.99|  1229.46|       1|           1229.46|           2294.99| 2821598|   Bronze|
|North America|United States|        Clothing|    49.99|    28.84|       6|            173.04|            299.94|   51901|   Bronze|
|North America|United States|        Clothing|    37.99|    22.79|   

In [23]:
b_df.groupby("Sales_Grp").agg(count("Region").alias("Total_Transactions")).sort(desc("Total_Transactions")).show(10)

+---------+------------------+
|Sales_Grp|Total_Transactions|
+---------+------------------+
|   Bronze|             53169|
|   Silver|              6694|
|     Gold|               588|
| Platinum|               469|
+---------+------------------+

